In [ ]:
%load_ext autoreload
%autoreload 2

import logging
from fabduckdb import register_function
from fabduckdb.table_functions import fab_functions
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

logging.basicConfig(
    format="%(asctime)s %(name)s %(levelname)-8s %(message)s",
    level=logging.DEBUG,
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("fab_duckdb")
logger.setLevel(logging.DEBUG)

In [ ]:
def df_creator(x: int) -> pd.DataFrame:
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2023, 12, 31)
    num_rows = x

    datetime_col = pd.date_range(start_date, end_date, periods=num_rows)

    # Generate random numeric values
    numeric_col1 = np.random.rand(num_rows)
    numeric_col2 = np.random.randint(1, 100, num_rows)

    # Generate random string values
    string_col1 = np.random.choice(["apple", "banana", "cherry"], num_rows)
    string_col2 = np.random.choice(["red", "green", "blue"], num_rows)

    # Create the pandas DataFrame
    data = {
        "DateTime": datetime_col,
        "Numeric1": numeric_col1,
        "Numeric2": numeric_col2,
        "String1": string_col1,
        "String2": string_col2,
    }

    df = pd.DataFrame(data)
    return df


display(df_creator(2))

In [ ]:
"""Example using a function"""

register_function("mydfcreator", df_creator, generates_filepath=False)

In [ ]:
"""Example using a lambda & DataFrame"""
register_function(
    "dfcreate",
    lambda rows, cols, con=None: pd.DataFrame(np.random.rand(rows, cols)),
    generates_filepath=False,
)

duckdb.connect().execute("select * from dfcreate(cols=4,rows=3)").df()

In [ ]:
"""Example returning a filepath to a parquet file"""

register_function(
    "dfcreate",
    lambda rows, cols, filename, con: pd.DataFrame(
        np.random.rand(rows, cols)
    ).to_parquet(filename),
    generates_filepath=True,
)

duckdb.connect().execute("select * from dfcreate(3,4)").df()

In [ ]:
"""Example returning a pyarrow table"""

register_function(
    "dfcreate_pa",
    lambda rows, cols, con=None: __import__("pyarrow").table(
        {
            f"col{i}": [
                "".join(
                    __import__("random").choices(
                        __import__("string").ascii_letters, k=5
                    )
                )
                for _ in range(rows)
            ]
            for i in range(cols)
        }
    ),
    generates_filepath=False,
)

duckdb.connect().execute("select * from dfcreate_pa(3,4)").df()

In [ ]:
register_function(
    "df_creator",
    lambda rows, cols, con: pd.DataFrame(np.random.rand(rows, cols)),
    generates_filepath=False,
)


fab_functions.extract_and_replace_functions("select * from mydfcreator(1,2)")

In [ ]:
register_function(
    "dfcreate",
    lambda rows, cols, filename, con: pd.DataFrame(
        np.random.rand(rows, cols)
    ).to_parquet(filename),
    generates_filepath=True,
)

duckdb.connect().execute("select * from dfcreate(3,4)").df()